# 🚀 Distill a tool-use agent into `Qwen/Qwen3.5-4B` with Sequence-level KD (SeqKD) on Amazon SageMaker AI

**Sequence-level Knowledge Distillation (SeqKD)** is behavior cloning: fine-tune a small student on a capable teacher's *decoded* trajectories (loss on the completion only). Here we distill agentic **tool-use** (shopping) into **Qwen/Qwen3.5-4B**.

**What you will learn**
1. Encode teacher ShoppingBench trajectories into `{prompt, completion}` pairs with the student's exact non-thinking chat template.
2. Run SeqKD (SFT on those pairs) as a SageMaker `ModelTrainer` training job.
3. Merge the LoRA adapter and evaluate task success (ASR) + MMLU/IFEval regression.

Reference results (thinking-off): **ASR 43.2 (base) → 62.4 (LoRA r64)**, MMLU ~0.697 and IFEval ~82.6 held.

In [ ]:
%pip install -Uq "sagemaker>=2.240" boto3 "datasets>=3.0.0" transformers huggingface_hub

## 00. Setup

In [ ]:
import json, boto3, sagemaker

region = boto3.Session().region_name
sess = sagemaker.Session(boto3.Session(region_name=region))
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()

PREFIX = "distillation/seqkd-shopping/qwen3.5-4b"
S3_ROOT = f"s3://{bucket}/{PREFIX}"
HF_STUDENT = "Qwen/Qwen3.5-4B"   # Apache-2.0, ungated; loaded inside the training container
print(region, role, S3_ROOT)

## 01. Data Prep — encode teacher trajectories → `{prompt, completion}`

One training example per assistant step; loss falls only on the completion. We render with the **student's own chat template** (`apply_chat_template`, `enable_thinking=False`, `tools=`) via a prefix-diff so `prompt + completion` is byte-identical to inference (non-thinking: prompt ends with a closed-empty `<think></think>`; tool calls render in Qwen-native format). TRL auto-detects the prompt/completion format and masks the prompt. Swap `REPO`/`sft_file` for your own teacher trajectories.

In [ ]:
import json, random
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import AutoTokenizer

REPO = "oro-ai/sn15-shoppingbench-sft-15k"   # example teacher trajectories
jsonls = [f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".jsonl")]
sft_file = next(f for f in jsonls if "metadata" not in f.lower() and "eval" not in f.lower())
raw = [json.loads(l) for l in open(hf_hub_download(REPO, sft_file, repo_type="dataset")) if l.strip()]
raw = [r for r in raw if r.get("messages")]
print(len(raw), "trajectories")

tok = AutoTokenizer.from_pretrained(HF_STUDENT, trust_remote_code=True)
SYSTEM = ("You are a shopping assistant. Use the available tools to satisfy the request: "
          "search the catalog, inspect candidates, verify the constraints, then recommend.")

def _render(msgs, tools, add_gen):
    return tok.apply_chat_template(msgs, tools=tools, tokenize=False,
                                   add_generation_prompt=add_gen, enable_thinking=False)

def to_rows(traj):
    tools = traj.get("tools")
    convo = [{"role": "system", "content": SYSTEM}] + [m for m in traj["messages"] if m.get("role") != "system"]
    rows = []
    for i, m in enumerate(convo):
        if m.get("role") != "assistant":
            continue
        prompt = _render(convo[:i],     tools, add_gen=True)
        full   = _render(convo[:i + 1], tools, add_gen=False)
        assert full.startswith(prompt), "chat-template prefix mismatch"
        completion = full[len(prompt):]
        if completion.strip():
            rows.append({"prompt": prompt, "completion": completion})
    return rows

examples = [r for traj in raw for r in to_rows(traj)]
random.seed(42); random.shuffle(examples)
val, train = examples[:1000], examples[1000:]
print(f"{len(raw)} trajectories -> {len(examples)} SFT examples  (train {len(train)} / val {len(val)})")

def write_jsonl(rows, p):
    with open(p, "w") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
write_jsonl(train, "seqkd_train.jsonl"); write_jsonl(val, "seqkd_val.jsonl")

#### Upload to S3 (ModelTrainer mounts each channel at `/opt/ml/input/data/<channel>/`)

In [ ]:
s3 = boto3.client("s3", region_name=region)
s3.upload_file("seqkd_train.jsonl", bucket, f"{PREFIX}/train/seqkd_train.jsonl")
s3.upload_file("seqkd_val.jsonl",   bucket, f"{PREFIX}/val/seqkd_val.jsonl")
TRAIN_S3, VAL_S3 = f"{S3_ROOT}/train/", f"{S3_ROOT}/val/"
print(TRAIN_S3, VAL_S3)

## 02. Fine-Tune (SeqKD) using SageMaker `ModelTrainer`

`scripts/train_seqkd.py` runs TRL `SFTTrainer` on the `{prompt, completion}` rows (LoRA or full).

**Why the HyperPod verl training image (not a stock PyTorch DLC):** Qwen3.5-4B is a **linear-attention / conv1d hybrid**, so the linear-attn path needs `flash-linear-attention`, which requires `triton>=3.7.1`. That collides with the image's `torch`-pinned `triton==3.6` under pip's resolver, so `train_seqkd.py` force-installs `triton==3.8.0` + `flash-linear-attention==0.5.2` with `--no-deps` at job start (`--install-kernels`, default on) and uses **`sdpa`** attention (`flash_attention_2` CUDA-IMAs on the managed driver). The launcher `sm_train.sh` installs `requirements_verl.txt` then runs `torchrun train_seqkd.py`.

In [ ]:
import time
from sagemaker.modules.configs import Compute, OutputDataConfig, SourceCode, StoppingCondition, InputData
from sagemaker.modules.train import ModelTrainer

# HyperPod verl-0.7.0 SMTJ recipe image (us-west-2): ships torch 2.10 + transformers + flash-attn; we add
# trl/peft/datasets via requirements_verl.txt and the fla/triton kernels via the --no-deps bootstrap.
IMAGE = "920498770698.dkr.ecr.us-west-2.amazonaws.com/hyperpod-recipes-verl-0-7-0:verl-v1.1.0-smtj"
INSTANCE_TYPE = "ml.p5.48xlarge"   # 8x H100; ml.p4de.24xlarge (A100) also validated

# LoRA r64 @ 4096 replicates the reference arm (ASR 62.4). Effective batch = 1 x grad-accum 2 x 8 GPUs = 16.
hp = [
    "--max-length", "4096", "--lora-r", "64", "--lr", "2e-5", "--epochs", "2",
    "--batch-size", "1", "--grad-accum", "2", "--fft", "false", "--attn", "sdpa",
]

source_code = SourceCode(
    source_dir="./scripts",
    command=f"bash sm_train.sh {' '.join(hp)}",
)
job_name = "qwen3-5-4b-shop-seqkd-r64"
model_trainer = ModelTrainer(
    training_image=IMAGE,
    source_code=source_code,
    base_job_name=job_name,
    compute=Compute(instance_type=INSTANCE_TYPE, instance_count=1,
                    keep_alive_period_in_seconds=1800, volume_size_in_gb=500),
    stopping_condition=StoppingCondition(max_runtime_in_seconds=36000),
    output_data_config=OutputDataConfig(s3_output_path=f"{S3_ROOT}/output/{job_name}"),
    role=role,
)
model_trainer.train(
    input_data_config=[
        InputData(channel_name="train", data_source=TRAIN_S3),
        InputData(channel_name="val",   data_source=VAL_S3),
    ],
    wait=False,
)

> **Tip — verify before the full run:** add `"--max-steps", "5", "--max-train-samples", "256", "--max-val-samples", "64"` to `hp` for a fast smoke test. The first `{'loss': ...}` log line confirms the kernel bootstrap + sdpa path work on your instance before you commit to the full ~1.5–3h run.

> **FFT arm (optional):** `--fft true --fsdp "shard_grad_op auto_wrap" --lr 1e-5` (full_shard crashes the hybrid's conv/linear-attn params → use shard_grad_op).

## 03. Merge the LoRA adapter

The LoRA run saves an adapter under the output prefix (`model.tar.gz`). Merge into the base for serving/eval (a 4B bf16 merge runs on CPU). The FFT run saves full weights — skip the merge.

In [ ]:
import os, glob, tarfile, subprocess
from transformers import AutoModelForImageTextToText, AutoTokenizer
from peft import PeftModel
from huggingface_hub import snapshot_download

ART = model_trainer._latest_training_job.model_artifacts.s3_model_artifacts   # s3://.../model.tar.gz
subprocess.run(["aws", "s3", "cp", ART, "model.tar.gz"], check=True)
with tarfile.open("model.tar.gz") as t: t.extractall("adapter")
adir = os.path.dirname(glob.glob("adapter/**/adapter_config.json", recursive=True)[0])

base = AutoModelForImageTextToText.from_pretrained(HF_STUDENT, torch_dtype="bfloat16", trust_remote_code=True)
PeftModel.from_pretrained(base, adir).merge_and_unload().save_pretrained("merged")
AutoTokenizer.from_pretrained(HF_STUDENT).save_pretrained("merged")
# vLLM needs these VLM configs; copy from the base snapshot if the merged dir misses them.
cfg = snapshot_download(HF_STUDENT, allow_patterns=["preprocessor_config.json", "processor_config.json"])
for f in ("preprocessor_config.json", "processor_config.json"):
    if os.path.exists(f"{cfg}/{f}"): subprocess.run(["cp", f"{cfg}/{f}", "merged/"])
subprocess.run(["aws", "s3", "cp", "--recursive", "merged", f"{S3_ROOT}/merged/r64/"], check=True)
print("merged -> ./merged and", f"{S3_ROOT}/merged/r64/")

## 04. Evaluate

See **`evaluate-tool-call-accuracy.ipynb`** for the base-vs-student comparison. Two axes:

- **Tool-use ASR (headline)** — ShoppingBench rollout scored by the task's success-rate metric. Reference: base **43.2 → 62.4** (LoRA r64).
- **Regression (MMLU + non-thinking IFEval)** — offline via `scripts/eval_lmeval.sh <MERGED_DIR> <IFACE> <GPU> <TAG>` (one Qwen3.5 vLLM at a time; MMLU `--batch_size 8 --enforce_eager`; IFEval served standalone with a non-thinking template + `lm_eval --apply_chat_template`). Reference: MMLU ~0.697, IFEval ~82.6 — both held.